# Corrected version of extended coupling of incoherent astronomical source

We now set the default integration range for the surface brightness * coupling efficiency weighted sum average integral to the fibre core radius as any rays incident on the cladding will be refracted, knonwn as leaky rays. 

We also now provide the option to input a 2D map to the function so that the coupling efficienc does not have to be computed for every single pixel within the extended source coupling efficiency funtion. 

In [1]:
import astropy.units as u
from astropy.modeling.models import Sersic2D
# Source: https://docs.astropy.org/en/stable/api/astropy.modeling.functional_models.Sersic2D.html

In [2]:
from matplotlib import pyplot as plt
import numpy as np
from scipy.special import jv, kv
from scipy.special import j1 # Bessel function of the first kind of order 1, for the diffraction pattern calculation
from scipy.optimize import root_scalar
from scipy.constants import c, epsilon_0, mu_0, pi
import scipy.integrate as integrate

In [3]:
from scipy.special import jn_zeros  # NEW: needed to get the first zero of J0

In [6]:
import sys
from pathlib import Path
import joblib
import time

src_path = Path("/Users/tliu/oh_suppression/codes/src")

if str(src_path) not in sys.path:
    sys.path.append(str(src_path))

from oh_suppression.coupling import *

In [7]:
from oh_suppression.point_source_coupling import *
from oh_suppression.flat_background_coupling import *
from oh_suppression.extended_source_coupling import *

In [8]:
from scipy.interpolate import RegularGridInterpolator
from scipy.interpolate import PchipInterpolator
from tqdm.auto import tqdm
import math

In [9]:
import sys
from pathlib import Path
import joblib
import time

src_path = Path("/Users/tliu/oh_suppression/codes/src")

if str(src_path) not in sys.path:
    sys.path.append(str(src_path))

from oh_suppression.coupling import *

In [ ]:
def sigma_jitter_milliarcsec_to_metres(sigma_jitter_rms, focal_length, demagnification): 
    """
    Convert the RMS jitter from milliarcseconds to metres at the fibre plane.

    Parameters: 
    sigma_jitter_rms : float
        RMS jitter in milliarcseconds.
    focal_length : float
        Focal length of the telescope in metres.
    demagnification : float 
        Demagnification factor from the telescope focal plane to the fibre plane, done by the reimaging optics. 

    Returns:
    float
        RMS jitter in metres at the fibre plane.
    """
    rad_to_arcsec = (180 / np.pi) * 3600 # essentially, 1 radian = 206264.80624709636 arcseconds

    # Convert jitter from milliarcseconds to radians
    sigma_theta_radians = sigma_jitter_rms * 1e-3 / rad_to_arcsec 

    # Convert jitter from radians to physical units (metres) at the fibre plane using the focal length and demagnification

    sigma_jitter_metres = demagnification * focal_length * sigma_theta_radians

    return sigma_jitter_metres

In [ ]:
# def metres_to_arcseconds(metres, plate_scale_arcsec_per_metre): 
#     """
#     Convert a physical distance in metres across the fibre plane to arcseconds on the sky.

#     Parameters: 
#     metres : float
#         Physical distance in metres at the fibre plane.
#     plate_scale_arcsec_per_metre : float
#         Plate scale in arcseconds per metre.

#     Returns:
#     float
#         Distance in arcseconds on the sky.
#     """

#     # Convert physical distance in metres at the fibre plane to angular distance in radians on the sky
#     # 1 radian = 1 metre / (focal_length * demagnification)
#     theta_arcseconds = metres * plate_scale_arcsec_per_metre

#     return theta_arcseconds

In [ ]:
# Write a helper function that reads the total coupling efficiency map and modal 
# coupling efficiency maps for each radii for the saved file format. 

# radius_range_file_path is the path to the file that contains the result for the radius range, 
# e.g. between V=2.405 and V_cutoff of LP02.
def read_coupling_efficiency_maps(radius_range_file_path): 
    for radius in radius_range_file_path: 
        total_coupling_efficiency_map 

In [ ]:
model_cent = Sersic2D(amplitude=1, r_eff = Re_arcsec, n = n_sersic, x_0=0, y_0=0, ellip=ellip, theta=np.radians(PA_deg))

In [ ]:
def extended_source_at_one_radius_2d(
        a,
        x, y, x_1d, y_1d, 
        NA, lam0, I_source, 
        F_eff,
        alpha, 
        eta_map_2d, 
        E_S=1.0, 
        az_sym=True, 
        plate_scale_arcsec_per_metre, # effective plate scale
        mode_case="few_mode", 
        decentre_max="core_radius", 
):
    """
    For one fibre core radius a: 
        1. Prepare fibre modes
        2. Calculate eta_psf(d)
        3. Integrate eta_psf(d) to get coupling effieciency for an extended source with a given brightness distribution

    Parameters:
    I_source: the intensity distribution of the extended source, 2d numpy array, same shape as x and y grids

    """


    # Choose decentre_max to be the cladding radius if not specified
    if decentre_max is None: 
        decentre_max = 125e-6 # in meters, this is the maximum radius of the grid we prepared for the field calculations, so we can't go beyond this

    elif decentre_max == "core_radius": 
        decentre_max = a # core radius

    # decentre_array = np.linspace(0, decentre_max, n_decentre)

    # # Calculate eta_psf(d) for the range of decentre values
    # eta_psf,  mode_results_decentre = eta_psf_vs_radial_decentre_position(
    #     prepared_modes, 
    #     x, y, x_1d, y_1d,
    #     lam0=lam0, F_eff=F_fibre, alpha=alpha,
    #     decentre_array=decentre_array,
    #     E_S=E_S
    # )

    # # Integrate over the extended source brightness distribution to get the coupling efficiency for the extended source
    # eta_map_2d=radial_efficiency_to_2d(decentre_array, eta_psf, x, y)

    # create the image according the the fibre core radius a

    # Convert fibre radius in metres to arcseconds
    
    # Convert fibre radius in metres to arcseconds

    theta_arcsec = a * plate_scale_arcsec_per_metre

    x_arcsec_1d = x_1d * plate_scale_arcsec_per_metre
    y_arcsec_1d = y_1d * plate_scale_arcsec_per_metre

    x_arcsec, y_arcsec = np.meshgrid(x_arcsec_1d, y_arcsec_1d)

    # Use predefined model to create the image of the extended source in arcseconds
    # not sure if i should define img inside or outside the function
    img = model_cent(x_arcsec, y_arcsec)

    eta_extended_source = eta_extended_source_2d(eta_map_2d=eta_map_2d,
        galaxy_brightness_2d=I_source, x_1d=x_1d, y_1d=y_1d)
    
    
    return {
        "a": a,
        "V": V,
        "F_eff": F_fibre,
        "eta_centre": eta_centre,
        "eta_extended_source": eta_extended_source,
        "decentre_array": decentre_array,
        "eta_psf": eta_psf,
        "mode_results_centred": mode_results_centred,
        "mode_results_decentre": mode_results_decentre,
        # "E_image_opt": E_image_opt,
        # "I_image_opt": I_image_opt
    }